In [1]:
# Data Collection 
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd
# Load the text data from the Gutenberg corpus
data = gutenberg.raw('shakespeare-hamlet.txt')

# Save the file to a local directory
with open('shakespeare_hamlet.txt', 'w') as file:
    file.write(data)


[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\MSI\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [2]:
# Data Preprocessing
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split


In [3]:
# Load the text data
import re
with open('shakespeare_hamlet.txt', 'r') as file:
    text= file.read().lower() # Read the text file and convert it to lowercase
    text= re.sub(r'[^\w\s]', '', text) # Remove punctuation from the text using a regular expression


# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text]) # Create a word index based on the tokenized text
total_words= len(tokenizer.word_index)+1 # Get the total number of unique words in the text
total_words

4800

In [4]:
# Create input sequences using the tokenized text
input_sequences= []
for line in text.split('\n'):
    token_list= tokenizer.texts_to_sequences([line])[0] # Convert the line to a sequence of integers
    for i in range(1, len(token_list)):
        n_gram_sequence= token_list[:i+1] # Create n-gram sequences
        input_sequences.append(n_gram_sequence)



In [5]:
# pad the sequences to ensure they are of the same length
max_sequence_length= max([len(x) for x in input_sequences]) # Get the maximum sequence length
input_sequences= np.array(pad_sequences(input_sequences, maxlen= max_sequence_length, padding='pre')) # Pad the sequences with zeros at the beginning
input_sequences


array([[   0,    0,    0, ...,    0,    1,  683],
       [   0,    0,    0, ...,    1,  683,    4],
       [   0,    0,    0, ...,  683,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1042],
       [   0,    0,    0, ...,   45, 1042,    4],
       [   0,    0,    0, ..., 1042,    4,  192]])

In [6]:
# Create predictors and labels
x,y= input_sequences[:,:-1], input_sequences[:,-1] # Split the sequences into predictors (x) and labels (y)


In [7]:
y=tf.keras.utils.to_categorical(y,num_classes=total_words) # Convert the labels to one-hot encoded vectors
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [8]:
# Train-Test Split
x_train, x_test, y_train, y_test= train_test_split(x,y, test_size= 0.2) # Split the data into training and testing sets
x_train.shape, y_train.shape, x_test.shape, y_test.shape


((20490, 13), (20490, 4800), (5123, 13), (5123, 4800))

In [9]:
# Define early stopping callback
from tensorflow.keras.callbacks import EarlyStopping
early_stopping= EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True) # Stop training if the validation loss does not improve for 10 consecutive epochs

In [10]:
# Train the LSTM RNN model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, GRU

model= Sequential()
model.add(Embedding(total_words, 100, input_shape= (max_sequence_length-1,))) # Add an embedding layer to convert the input sequences into dense vectors of fixed size
model.add(GRU(150, return_sequences= True)) # Add a GRU layer with 150 units to capture the temporal dependencies in the data
model.add(Dropout(0.2)) # Add a dropout layer to prevent overfitting
model.add(GRU(100)) # Add another GRU layer with 100 units
model.add(Dense(total_words, activation='softmax')) # Add a dense output layer with softmax activation to predict the next word in the sequence

# Compile the model
opt= tf.keras.optimizers.Adam(learning_rate=0.001) # Create an Adam optimizer with a learning rate of 0.001
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy']) # Compile the model with categorical cross-entropy loss and the Adam optimizer
model.summary() # Print the model summary to see the architecture of the model


c:\Users\MSI\python\RNN_Project\venv\lib\site-packages\keras\src\layers\core\embedding.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 13, 100)        │       480,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 13, 150)        │       113,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 13, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 100)            │        75,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4800)           │       484,800 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,153,800 (4.40 MB)

 Trainable params: 1,153,800 (4.40 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# Train the model

history= model.fit(x_train, y_train, epochs=50,validation_data= (x_test, y_test), verbose=1, callbacks=[early_stopping]) # Train the model for 50 epochs and validate on the test set



Epoch 1/50
641/641 ━━━━━━━━━━━━━━━━━━━━ 25s 31ms/step - accuracy: 0.0266 - loss: 7.1656 - val_accuracy: 0.0359 - val_loss: 6.8796
Epoch 2/50
641/641 ━━━━━━━━━━━━━━━━━━━━ 19s 30ms/step - accuracy: 0.0354 - loss: 6.4748 - val_accuracy: 0.0482 - val_loss: 6.8680
Epoch 3/50
641/641 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.0521 - loss: 6.2205 - val_accuracy: 0.0580 - val_loss: 6.8669
Epoch 4/50
641/641 ━━━━━━━━━━━━━━━━━━━━ 18s 28ms/step - accuracy: 0.0619 - loss: 5.9708 - val_accuracy: 0.0681 - val_loss: 6.8498
Epoch 5/50
641/641 ━━━━━━━━━━━━━━━━━━━━ 19s 30ms/step - accuracy: 0.0725 - loss: 5.7506 - val_accuracy: 0.0730 - val_loss: 6.9408
Epoch 6/50
641/641 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - accuracy: 0.0881 - loss: 5.5000 - val_accuracy: 0.0709 - val_loss: 7.0501
Epoch 7/50
641/641 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - accuracy: 0.0936 - loss: 5.2651 - val_accuracy: 0.0753 - val_loss: 7.1404
Epoch 8/50
641/641 ━━━━━━━━━━━━━━━━━━━━ 18s 27ms/step - accuracy: 0.1060 - loss: 4.9938 - 

In [12]:
# Function to predict the next word in a given text sequence

def predict_next_word(model, tokenizer, text_sequence, max_sequence_length):
    token_list= tokenizer.texts_to_sequences([text_sequence])[0]    # Convert the input text sequence to a sequence of integers
    if len(token_list)>= max_sequence_length:
        token_list= token_list[-(max_sequence_length-1):] # If the input sequence is longer than the maximum sequence length, truncate it to fit the model's input shape
    token_list= pad_sequences([token_list], maxlen= max_sequence_length-1, padding='pre') # Pad the sequence with zeros at the beginning
    prediction= model.predict(token_list, verbose=0)# Predict the next word using the trained model

    predicted_word_index= np.argmax(prediction, axis=1) # Get the index of the predicted word with the highest probability
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None # Return None if no word is found for the predicted index







In [13]:
# Example usage of the predict_next_word function

input_text= "To be or not to be"
print(f"Input Text: {input_text}")
max_sequence_length= model.input_shape[1] + 1 # Get the maximum sequence length from the model's input shape
predicted_word= predict_next_word(model,tokenizer,input_text,max_sequence_length)
print(f"Predicted Next Word: {predicted_word}")


Input Text: To be or not to be
Predicted Next Word: not


In [14]:
# Save the trained model

model.save("gru_next_word_prediction_model.keras")

# Save the tokenizer for future use

import pickle
with open('tokenizer_GRU.pickle', 'wb') as file:
    pickle.dump(tokenizer, file, protocol= pickle.HIGHEST_PROTOCOL)
    # protocol= pickle.HIGHEST_PROTOCOL ensures that the highest available protocol is used for pickling, which can improve performance and reduce file size when saving the tokenizer object.



In [15]:
# Example usage of the predict_next_word function

input_text= "And let vs heare "
print(f"Input Text: {input_text}")
max_sequence_length= model.input_shape[1] + 1 # Get the maximum sequence length from the model's input shape
predicted_word= predict_next_word(model,tokenizer,input_text,max_sequence_length)
print(f"Predicted Next Word: {predicted_word}")

Input Text: And let vs heare 
Predicted Next Word: you
